In [ ]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize
import os

In [4]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

In [ ]:
#Step 4: TCA Functions

class PostTradeTCA:
    @staticmethod
    def IS_PaperReturn(side, S, Pd, Pn):
        """Calculates Paper Return component of Implementation Shortfall"""
        return side * S * (Pn - Pd)
    
    @staticmethod
    def implementation_shortfall(side, S, X, Pd, P0, Pn, Pavg, commission_per_share):
        """
        Calculates Implementation Shortfall (IS) and its components.
        """
        R = S - X 
        fixed_cost = X * commission_per_share 
        
        # Paper and Portfolio Returns
        paper_return = side * S * (Pn - Pd)
        portfolio_return = (side * X * (Pn - Pavg)) - fixed_cost
        
        # IS Calculation
        is_total = paper_return - portfolio_return
        
        # Cost Decomposition
        delay_cost = side * S * (P0 - Pd)
        execution_cost = side * X * (Pavg - P0)
        opportunity_cost = side * R * (Pn - P0)
        
        return {
            "Paper Return": paper_return,
            "Portfolio Return": portfolio_return,
            "IS Total": is_total,
            "Delay Cost": delay_cost,
            "Execution Cost": execution_cost,
            "Opportunity Cost": opportunity_cost,
            "Fixed Cost": fixed_cost
        }

    @staticmethod
    def arrival_cost_bp(side, Pavg, P0):
        """Calculates Arrival Cost in basis points"""
        return side * ((Pavg - P0) / P0) * 10000

    @staticmethod
    def vwap_slippage_bp(side, Pavg, VWAP):
        """Calculates VWAP Slippage in basis points"""
        return side * ((Pavg - VWAP) / VWAP) * 10000

    @staticmethod
    def benchmark_cost_bp(side, Pavg, Pb):
        """Calculates Benchmark Cost for a specified benchmark price Pb"""
        return side * ((Pavg - Pb) / Pb) * 10000

    @staticmethod
    def value_add(expected_cost, actual_arrival_cost, timing_risk):
        """
        Calculates Broker Value-Add and Z-Score.
        Note: expected_cost is MI + PA.
        """
        value_add = expected_cost - actual_arrival_cost
        z_score = value_add / timing_risk
        return {"Value Add": value_add, "Z-Score": z_score}

    @staticmethod
    def calculate_rpm(side, p_avg, trade_data):
        """
        Calculates Relative Performance Measure (RPM).
        trade_data: List of tuples (price, volume)
        """
        total_volume = sum(v for p, v in trade_data)
        
        if side == 1: # Buy Order
            vol_above = sum(v for p, v in trade_data if p > p_avg)
            vol_at = sum(v for p, v in trade_data if p == p_avg)
            rpm = (vol_above + (0.5 * vol_at)) / total_volume
        else: # Sell Order
            vol_below = sum(v for p, v in trade_data if p < p_avg)
            vol_at = sum(v for p, v in trade_data if p == p_avg)
            rpm = (vol_below + (0.5 * vol_at)) / total_volume
            
        return rpm * 100 # Expressed as percentage

In [13]:
tca = PostTradeTCA()

# --------------------------------------
# Homework 6
# --------------------------------------

## Implementation Shortfall

### Q1

In [52]:
IS_results = tca.implementation_shortfall(
    side= 1, # Buy Order = 1, Sell Order = -1
    S = 200000, 
    X = 180000, 
    Pd = 60.00, 
    P0 = 60.05, 
    Pn = 60.65, 
    Pavg = 60.45, 
    commission_per_share = 0.01 
)

print("--- Implementation Shortfall Analysis ---")
for key, value in IS_results.items():
    print(f"{key:18}: ${value:,.2f}")

--- Implementation Shortfall Analysis ---
Paper Return      : $130,000.00
Portfolio Return  : $34,200.00
IS Total          : $95,800.00
Delay Cost        : $10,000.00
Execution Cost    : $72,000.00
Opportunity Cost  : $12,000.00
Fixed Cost        : $1,800.00


## Arrival Cost

### Q2

In [44]:
cost = tca.arrival_cost_bp(
    side = 1, 
    Pavg = 25.30, 
    P0 = 25.15
)

print("--- Arrival Cost Analysis ---")
print(f"Arrival Cost: {cost:.4f} bp")

--- Arrival Cost Analysis ---
Arrival Cost: 59.6421 bp


## VWAP Slippage

### Q3

In [ ]:
vwap_slippage = tca.vwap_slippage_bp(
    side = -1, 
    Pavg = 55.37, 
    VWAP = 55.35
)

print("--- VWAP Slippage Analysis ---")
print(f"VWAP Slippage: {vwap_slippage:.5f} bp")

--- VWAP Slippage Analysis ---
VWAP Slippage: -3.61337 bp


## Benchmark Cost

### Q4

In [45]:
benchmark_cost = tca.benchmark_cost_bp(
    side = 1, 
    Pavg = 150.65, 
    Pb = 150.95
)

print("--- Benchmark Cost Analysis ---")
print(f"Benchmark Cost: {benchmark_cost:.5f} bp")

--- Benchmark Cost Analysis ---
Benchmark Cost: -19.87413 bp


## Value-Add

### Q5

In [ ]:
arrival_cost = tca.arrival_cost_bp(
    side = 1, 
    Pavg = 135.35,
    P0 = 134.90
)

value_add = tca.value_add(
    expected_cost = 30, # Expected cost (MI + PA) in basis points
    actual_arrival_cost = arrival_cost,
    timing_risk = 40
)

print("--- Value Add Analysis (Q5) ---")
# print(f"Arrival Cost: {arrival_cost:.4f} bp")
print(f"Value Add:    {value_add['Value Add']:.4f} bp")
print(f"Z-Score:      {value_add['Z-Score']:.4f}")

--- Value Add Analysis (Q5) ---
Value Add:    -3.3580 bp
Z-Score:      -0.0840


### Q6

In [ ]:
arrival_cost = tca.arrival_cost_bp(
    side = 1, 
    Pavg = 51.22,
    P0 = 51.00
)

value_add = tca.value_add(
    expected_cost = 35, # Expected cost (MI + PA) in basis points
    actual_arrival_cost = arrival_cost,
    timing_risk = 65
)

print("--- Value Add Analysis (Q5) ---")
# print(f"Arrival Cost: {arrival_cost:.4f} bp")
print(f"Value Add:    {value_add['Value Add']:.4f} bp")
print(f"Z-Score:      {value_add['Z-Score']:.4f}")

--- Value Add Analysis (Q5) ---
Value Add:    -8.1373 bp
Z-Score:      -0.1252


## RPM

### Q7

In [51]:
rpm_data= [
    (24.50, 5000), 
    (26.50, 2500), 
    (27.50, 9000), 
    (25.25, 4000), 
    (25.50, 5000), 
    (26.00, 3000), 
    (25.00, 5000), 
    (27.10, 5000), 
    (24.00, 9000), 
    (24.75, 2500)
]

buy_rpm = tca.calculate_rpm(
    side = 1, 
    p_avg= 25.50, 
    trade_data = rpm_data
)

sell_rpm = tca.calculate_rpm(
    side = -1, 
    p_avg= 24.95, 
    trade_data = rpm_data
)

print("--- RPM Analysis ---")
print(f"Buy RPM: {buy_rpm:.5f} bp")
print(f"Sell RPM: {sell_rpm:.5f} bp")

--- RPM Analysis ---
Buy RPM: 44.00000 bp
Sell RPM: 33.00000 bp


### Q8

In [55]:
rpm_data= [
    (65.15, 25000),
    (65.30, 85000),
    (65.35, 35000),
    (65.25, 100000),
    (65.45, 40000),
    (65.00, 15000),
    (65.40, 50000),
    (65.50, 5000),
    (65.10, 80000),
    (65.20, 65000),
    (65.05, 125000)
]

buy_rpm = tca.calculate_rpm(
    side = 1, 
    p_avg= 65.25, 
    trade_data = rpm_data
)

sell_rpm = tca.calculate_rpm(
    side = -1, 
    p_avg= 65.33, 
    trade_data = rpm_data
)

print("--- RPM Analysis ---")
print(f"Buy RPM: {buy_rpm:.5f} bp")
print(f"Sell RPM: {sell_rpm:.5f} bp")

--- RPM Analysis ---
Buy RPM: 42.40000 bp
Sell RPM: 79.20000 bp
